# 2. Model Training

This notebook trains the 5-tier HP prediction models using engineered features.

**Input:** `helper_files/engineered_features.parquet`  
**Output:** `pickled_models/hp_model_cr*.pkl`

## Imports and Configs

In [15]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
from sklearn.preprocessing import StandardScaler


In [16]:
# Detect execution context and set paths dynamically
sys.path.insert(0, '.')

# Get the current working directory
cwd = Path.cwd()

# Check if we're in the notebooks directory or project root
if cwd.name == 'notebooks':
    # Running from notebooks directory (in Jupyter)
    DATA_DIR = '../data'
    PICKLED_MODELS_DIR = '../pickled_models'
    MONSTER_BUILDER_DIR = '../monster-builder-v2'
    HELPERS_DIR = './helper_files'
    IO_DIR = './notebooks_io'
    IN_NB_DIR = False
else:
    # Running from project root (via run_three_tier_model.py)
    DATA_DIR = './data'
    PICKLED_MODELS_DIR = './pickled_models'
    MONSTER_BUILDER_DIR = './monster-builder-v2'
    HELPERS_DIR = './notebooks/helper_files'
    IO_DIR = './notebooks/notebooks_io'
    IN_NB_DIR = False

print(f"📁 Execution context detected:")
print(f"   Current directory: {cwd}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Models directory: {PICKLED_MODELS_DIR}")

print("Imports successful")

📁 Execution context detected:
   Current directory: /workspaces/matrix_v0
   Data directory: ./data
   Models directory: ./pickled_models
Imports successful


In [17]:
# Add helper_files to path
if IN_NB_DIR is True:
    from helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
    )

    print("Imports successful")
else:
    from notebooks.helper_files import (
        get_phase3_features,
        train_constrained_model,
        ConstrainedModel,
        calculate_r2,
        calculate_mae,
        save_model,
        summarize_model_performance,
        extract_family,
    )  

    print("Imports successful")


Imports successful


## Load Engineered Features

In [18]:
# Load engineered features
load_path = IO_DIR + "/engineered_features.parquet"
df = pd.read_parquet(load_path)
print(f"Loaded {len(df)} monsters with {len(df.columns)} features")

Loaded 324 monsters with 130 features


In [19]:
# Get Phase 3 features
phase3_features = get_phase3_features()
print(f"Phase 3 features: {len(phase3_features)}")

Phase 3 features: 38


## Split by CR Tier

In [20]:
# Split by CR tier
df_cr1 = df[df['cr_tier'] == 'cr1'].copy()
df_cr2 = df[df['cr_tier'] == 'cr2'].copy()
df_cr3 = df[df['cr_tier'] == 'cr3'].copy()
df_cr4 = df[df['cr_tier'] == 'cr4'].copy()
df_cr5 = df[df['cr_tier'] == 'cr5'].copy()

print(f"CR < 1:    {len(df_cr1)} monsters")
print(f"CR 1-4:    {len(df_cr2)} monsters")
print(f"CR 5-10:   {len(df_cr3)} monsters")
print(f"CR 11-16:  {len(df_cr4)} monsters")
print(f"CR > 16:   {len(df_cr5)} monsters")

CR < 1:    113 monsters
CR 1-4:    99 monsters
CR 5-10:   65 monsters
CR 11-16:  27 monsters
CR > 16:   20 monsters


## Train/Test Strategy

**Matching original notebook behavior:** Using ALL data for both training and testing.
This measures model fit rather than generalization, which is intentional for this use case.

In [21]:
# Use ALL data for both training and testing (matching original notebook)
# This measures fit rather than generalization, which is intentional

def split_by_tier(df_tier, tier_name):
    """Return same data for both train and test (no split)."""
    print(f"  {tier_name}: Using all {len(df_tier)} samples for both training and testing")
    return df_tier, df_tier

In [22]:
# Split each tier
print("Splitting data:")
train_cr1, test_cr1 = split_by_tier(df_cr1, 'CR < 1')
train_cr2, test_cr2 = split_by_tier(df_cr2, 'CR 1-4')
train_cr3, test_cr3 = split_by_tier(df_cr3, 'CR 5-10')
train_cr4, test_cr4 = split_by_tier(df_cr4, 'CR 11-16')
train_cr5, test_cr5 = split_by_tier(df_cr5, 'CR > 16')

Splitting data:
  CR < 1: Using all 113 samples for both training and testing
  CR 1-4: Using all 99 samples for both training and testing
  CR 5-10: Using all 65 samples for both training and testing
  CR 11-16: Using all 27 samples for both training and testing
  CR > 16: Using all 20 samples for both training and testing


In [23]:
df_cr1.shape

(113, 130)

## Train Models

In [24]:
def train_tier_model(train_df, test_df, tier_name, phase3_features):
    """Train a model for a single CR tier."""
    print(f"\n{'='*60}")
    print(f"Training {tier_name} model...")
    print(f"{'='*60}")
    
    # Prepare features
    X_train = train_df[phase3_features].fillna(0).values
    y_train = train_df['residual_hp'].values
    
    X_test = test_df[phase3_features].fillna(0).values
    y_test = test_df['residual_hp'].values
    
    # Scale features (with_mean=False to preserve zero values)
    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train constrained model
    coefficients, intercept = train_constrained_model(
        X_train_scaled, y_train, phase3_features, scaler
    )
    
    # Create model object
    model = ConstrainedModel(coefficients, intercept)
    
    # Evaluate on test set
    y_pred_residual = model.predict(X_test_scaled)
    
    # Calculate full HP predictions
    y_pred_hp = test_df['hp_after_phase2'].values + y_pred_residual
    y_actual_hp = test_df['actual_hp'].values
    
    # Calculate metrics
    r2 = calculate_r2(y_actual_hp, y_pred_hp)
    mae = calculate_mae(y_actual_hp, y_pred_hp)
    
    print(f"\n{tier_name} Results:")
    print(f"   Training samples: {len(y_train)}")
    print(f"   Test R²:  {r2:.4f}")
    print(f"   Test MAE: {mae:.2f} HP")
    
    return {
        'model': model,
        'scaler': scaler,
        'train_count': len(y_train),
        'test_r2': r2,
        'test_mae': mae,
    }

In [25]:
# Train all models
results = {}

results['cr1'] = train_tier_model(train_cr1, test_cr1, 'CR < 1', phase3_features)
results['cr2'] = train_tier_model(train_cr2, test_cr2, 'CR 1-4', phase3_features)
results['cr3'] = train_tier_model(train_cr3, test_cr3, 'CR 5-10', phase3_features)
results['cr4'] = train_tier_model(train_cr4, test_cr4, 'CR 11-16', phase3_features)
results['cr5'] = train_tier_model(train_cr5, test_cr5, 'CR > 16', phase3_features)


Training CR < 1 model...

CR < 1 Results:
   Training samples: 113
   Test R²:  0.4097
   Test MAE: 4.98 HP

Training CR 1-4 model...

CR 1-4 Results:
   Training samples: 99
   Test R²:  0.5658
   Test MAE: 11.48 HP

Training CR 5-10 model...

CR 5-10 Results:
   Training samples: 65
   Test R²:  0.4799
   Test MAE: 18.28 HP

Training CR 11-16 model...



CR 11-16 Results:
   Training samples: 27
   Test R²:  0.9807
   Test MAE: 2.98 HP

Training CR > 16 model...

CR > 16 Results:
   Training samples: 20
   Test R²:  0.9533
   Test MAE: 16.04 HP


In [26]:
# Summary
summarize_model_performance(results)


5-BUCKET HP MODEL TRAINING COMPLETE

MODEL PERFORMANCE SUMMARY:

   CR < 1 Model:
      Training samples: 113
      Test R²:  0.4097
      Test MAE: 4.98 HP

   CR 1-4 Model:
      Training samples: 99
      Test R²:  0.5658
      Test MAE: 11.48 HP

   CR 5-10 Model:
      Training samples: 65
      Test R²:  0.4799
      Test MAE: 18.28 HP

   CR 11-16 Model:
      Training samples: 27
      Test R²:  0.9807
      Test MAE: 2.98 HP

   CR > 16 Model:
      Training samples: 20
      Test R²:  0.9533
      Test MAE: 16.04 HP

All 5 models trained successfully!


## Save Models

In [27]:
# Ensure output directory exists
os.makedirs(PICKLED_MODELS_DIR, exist_ok=True)
save_path = PICKLED_MODELS_DIR + "/hp_model_tier.pkl"
# Save each model
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:

    filepath = save_path.replace('tier', tier)
    save_model(
        results[tier]['model'],
        results[tier]['scaler'],
        phase3_features,
        filepath
    )
    print(f"Saved {tier} model to {filepath}")

print("\nAll models saved successfully!")

Saved cr1 model to ./pickled_models/hp_model_cr1.pkl
Saved cr2 model to ./pickled_models/hp_model_cr2.pkl
Saved cr3 model to ./pickled_models/hp_model_cr3.pkl
Saved cr4 model to ./pickled_models/hp_model_cr4.pkl
Saved cr5 model to ./pickled_models/hp_model_cr5.pkl

All models saved successfully!


In [28]:
# Display top feature coefficients for each tier
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    print(f"\n{tier.upper()} Top Features by Coefficient:")
    coefs = results[tier]['model'].coef_
    coef_df = pd.DataFrame({
        'feature': phase3_features,
        'coefficient': coefs
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(coef_df.head(10).to_string(index=False))


CR1 Top Features by Coefficient:
                 feature  coefficient
     speed_fly_deviation     2.434494
condition_immunity_count     2.356248
  speed_ground_deviation     2.280135
     total_ability_count    -1.654340
      inflicts_petrified    -1.512433
      inflicts_paralyzed    -1.119809
     vulnerability_count    -0.944148
       spellcaster_level    -0.886919
  save_proficiency_count     0.787243
             speed_climb    -0.731597

CR2 Top Features by Coefficient:
                    feature  coefficient
          spellcaster_level    -5.427423
     save_proficiency_count     5.154580
    skill_proficiency_count    -3.644197
        vulnerability_count     3.563618
                speed_climb    -3.278089
           inflicts_charmed    -3.092256
                 speed_swim     3.061025
                trait_count     2.596074
has_magic_resistance_scaled    -2.564761
               speed_burrow    -2.461672

CR3 Top Features by Coefficient:
                 feature  coe